# TATTVA-1 Polarity Encoder
Trains SciBERT to classify scientific claim relationships:
SUPPORTS / NEUTRAL / CONTRADICTS

**Training data:** 73,596 labeled pairs (MultiNLI + SciTail)
**Base model:** allenai/scibert_scivocab_uncased
**Time:** ~2 hours on T4 GPU (free Colab)
**Output:** polarity_encoder.pt

In [ ]:
# Install dependencies
!pip install transformers datasets torch -q
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('CUDA:', torch.cuda.is_available())

In [ ]:
# Upload training data from your Mac
# Run this on your Mac first:
# python3 -c "
# import sqlite3, json
# db = sqlite3.connect('knowledge_base/training_data.db')
# rows = db.execute('SELECT premise, hypothesis, label FROM nli_pairs').fetchall()
# json.dump([{'premise':r[0],'hypothesis':r[1],'label':r[2]} for r in rows], open('nli_training.json','w'))
# print('Saved', len(rows), 'examples to nli_training.json')
# "
# Then upload nli_training.json to Colab

from google.colab import files
uploaded = files.upload()  # Upload nli_training.json

In [ ]:
import json, torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
import numpy as np

# Load data
data = json.load(open('nli_training.json'))
print(f'Loaded {len(data)} examples')

# Split train/val
import random
random.shuffle(data)
split = int(len(data) * 0.9)
train_data = data[:split]
val_data = data[split:]
print(f'Train: {len(train_data)} | Val: {len(val_data)}')

In [ ]:
# Dataset class
class PolarityDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        enc = self.tokenizer(
            item['premise'], item['hypothesis'],
            max_length=self.max_len,
            truncation=True, padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label': torch.tensor(item['label'], dtype=torch.long)
        }

# Load SciBERT
print('Loading SciBERT...')
MODEL_NAME = 'allenai/scibert_scivocab_uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'Model loaded on {device}')

In [ ]:
# Create dataloaders
train_ds = PolarityDataset(train_data, tokenizer)
val_ds = PolarityDataset(val_data, tokenizer)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=len(train_loader)*3)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')

In [ ]:
# Training loop
import time

EPOCHS = 3
best_val_acc = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    t0 = time.time()
    
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds = outputs.logits.argmax(-1)
        correct += (preds == labels).sum().item()
        total += len(labels)
        
        if batch_idx % 100 == 0:
            elapsed = time.time() - t0
            print(f'Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.3f} | Acc: {correct/total:.3f} | Time: {elapsed:.0f}s')
    
    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(-1)
            val_correct += (preds == labels).sum().item()
            val_total += len(labels)
    
    val_acc = val_correct / val_total
    train_acc = correct / total
    print(f'\nEpoch {epoch+1} COMPLETE | Train Acc: {train_acc:.3f} | Val Acc: {val_acc:.3f}')
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'polarity_encoder_best.pt')
        print(f'New best model saved! Val acc: {val_acc:.3f}')

print(f'\nTraining complete. Best val accuracy: {best_val_acc:.3f}')

In [ ]:
# Save final model + tokenizer
model.load_state_dict(torch.load('polarity_encoder_best.pt'))
model.save_pretrained('tattva_polarity_encoder')
tokenizer.save_pretrained('tattva_polarity_encoder')
print('Model saved to tattva_polarity_encoder/')

# Download
import shutil
shutil.make_archive('tattva_polarity_encoder', 'zip', 'tattva_polarity_encoder')
files.download('tattva_polarity_encoder.zip')
print('Downloaded!')

In [ ]:
# Test the model
def predict_polarity(premise, hypothesis):
    model.eval()
    enc = tokenizer(premise, hypothesis, return_tensors='pt', max_length=256, truncation=True).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    pred = logits.argmax(-1).item()
    labels = ['SUPPORTS', 'NEUTRAL', 'CONTRADICTS']
    probs = torch.softmax(logits, -1)[0].tolist()
    return labels[pred], {l:round(p,3) for l,p in zip(labels,probs)}

# Test cases
tests = [
    ('LoRA reduces memory by 10000x during fine-tuning', 'LoRA significantly reduces GPU memory requirements'),
    ('LoRA works well at 7B scale', 'LoRA underperforms full fine-tuning at 70B on H100s'),
    ('Attention mechanism processes tokens in parallel', 'The weather today is sunny'),
]

for premise, hypothesis in tests:
    label, probs = predict_polarity(premise, hypothesis)
    print(f'P: {premise[:50]}')
    print(f'H: {hypothesis[:50]}')
    print(f'=> {label} | {probs}')
    print()